# Ablation 3 Counterfactual: Single-Scale + RandAugment (Production)

**Research Question**: Open: does single-scale training (512x512) with RandAugment improve wound segmentation.

**Intervention**: Train with single resolution (imgsz=512) and explicit RandAugment, all other augmentations default.

**Hypothesis**: the v5 95% collapse came from multi-scale conflicting with RandAugment. Single-scale with RandAugment should work.

**Expected Outcome**: Dice ≥78% (avoiding collapse, possibly matching or exceeding baseline)

**Status**: Production deployment (3 seeds, 100 epochs, P100 GPU)


In [ ]:
# Lock NumPy 1.26.4 (ultralytics upgrades to 2.x, breaks OpenCV)
import subprocess
import sys

print("Installing ultralytics...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)

print("Force-reinstalling NumPy 1.26.4...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "numpy==1.26.4"
], check=True)

import numpy as np
print(f"OK NumPy locked: {np.__version__}")
assert np.__version__.startswith("1.26"), f"NumPy version mismatch: {np.__version__}"


In [ ]:
import os
from pathlib import Path
import shutil

# Kaggle paths
IN_KAGGLE = os.path.exists("/kaggle/input")
BASE_INPUT_PATH = Path("/kaggle/input") if IN_KAGGLE else Path(".")
OUTPUT_PATH = Path("/kaggle/working") if IN_KAGGLE else Path(".")

# Dataset paths (FULL dataset)
DATASET_PATH = BASE_INPUT_PATH / "baseline-outputs" / "data_wound_seg"
CODE_PATH = BASE_INPUT_PATH / "wound-seg-code-v1"

# Output directory
ABLATION_OUTPUT = OUTPUT_PATH / "ablation_3_counterfactual_production"
ABLATION_OUTPUT.mkdir(exist_ok=True, parents=True)

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Local'}")
print(f"Dataset: {DATASET_PATH}")
print(f"Code: {CODE_PATH}")
print(f"Output: {ABLATION_OUTPUT}")


In [ ]:
# Copy dataset to working directory
WORKING_DATASET = OUTPUT_PATH / "data_wound_seg"

if not WORKING_DATASET.exists():
    print(f"Copying dataset from {DATASET_PATH}...")
    shutil.copytree(DATASET_PATH, WORKING_DATASET)
    print(f"OK Dataset copied to {WORKING_DATASET}")
else:
    print(f"OK Dataset already exists at {WORKING_DATASET}")

# Verify structure
required_dirs = ["train_images", "train_masks", "test_images", "test_masks"]
for d in required_dirs:
    path = WORKING_DATASET / d
    count = len(list(path.glob("*"))) if path.exists() else 0
    print(f"  {d}: {count} files")


In [ ]:
import sys
sys.path.insert(0, str(CODE_PATH))

print(f"OK Added {CODE_PATH} to Python path")
print(f"sys.path[0]: {sys.path[0]}")


In [ ]:
from ultralytics import YOLO
import yaml
import json
from datetime import datetime

print("OK Imports successful")


In [ ]:
# Create YOLO dataset configuration
config_path = OUTPUT_PATH / "wound_dataset.yaml"
config_content = f"""path: {WORKING_DATASET.parent}
train: data_wound_seg/train
val: data_wound_seg/test

names:
  0: wound
"""

with open(config_path, 'w') as f:
    f.write(config_content.strip())

print(f"OK YOLO config created: {config_path}")

# Verify config
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)
    print(f"  Dataset path: {config['path']}")
    print(f"  Train: {config['train']}")
    print(f"  Val: {config['val']}")


In [ ]:
# Training configuration
RANDOM_SEEDS = [42, 123, 456]  # 3 seeds for production
EPOCHS = 100
PATIENCE = 10

# P100-optimized hyperparameters
BATCH = 32
LR0 = 0.015
WORKERS = 8
AMP = False

# COUNTERFACTUAL: Single-scale + RandAugment
IMGSZ = 512  # Single resolution (NOT multi-scale)
AUTO_AUGMENT = "randaugment"  # Explicit RandAugment

print("Training Configuration (Counterfactual Production):")
print(f"  Seeds: {RANDOM_SEEDS}")
print(f"  Epochs: {EPOCHS} (patience: {PATIENCE})")
print(f"  Batch: {BATCH}")
print(f"  Learning rate: {LR0}")
print(f"  Workers: {WORKERS}")
print(f"  AMP: {AMP}")
print(f"  Image size: {IMGSZ} (SINGLE-SCALE)")
print(f"  Auto-augment: {AUTO_AUGMENT} (RandAugment)")


In [ ]:
# Multi-seed training with single-scale + RandAugment
training_results = {}

for seed in RANDOM_SEEDS:
    print(f"\n{'='*60}")
    print(f"Training seed {seed}")
    print(f"{'='*60}")

    # Initialize model
    model = YOLO("yolov8n-seg.pt")

    # Train with MINIMAL parameters (let YOLOv8 use defaults for other augmentations)
    results = model.train(
        data=str(config_path),
        epochs=EPOCHS,
        imgsz=IMGSZ,  # Single-scale: 512
        batch=BATCH,
        lr0=LR0,
        workers=WORKERS,
        patience=PATIENCE,
        amp=AMP,
        seed=seed,
        save=True,
        project=str(ABLATION_OUTPUT / "yolo_training"),
        name=f"seed_{seed}",
        exist_ok=True,
        verbose=False,

        # ONLY specify the intervention parameter
        auto_augment=AUTO_AUGMENT,  # Explicit RandAugment
        # All other augmentations use YOLOv8 defaults
    )

    # Save best model path
    best_model_path = ABLATION_OUTPUT / "yolo_training" / f"seed_{seed}" / "weights" / "best.pt"
    training_results[seed] = {
        "best_model_path": str(best_model_path),
        "completed": datetime.now().isoformat()
    }

    print(f"OK Seed {seed} complete: {best_model_path}")

print(f"\n{'='*60}")
print(f"All {len(RANDOM_SEEDS)} seeds completed")
print(f"{'='*60}")


## args.yaml Verification

**Check**: YOLOv8 used single-scale with RandAugment for all seeds.


In [ ]:
import yaml

print("\n=== PARAMETER VERIFICATION ===\n")

for seed in RANDOM_SEEDS:
    args_path = ABLATION_OUTPUT / "yolo_training" / f"seed_{seed}" / "args.yaml"

    with open(args_path, 'r') as f:
        args = yaml.safe_load(f)

    print(f"Seed {seed} - Checking critical parameters:")

    # Verify single-scale
    imgsz = args.get('imgsz')
    print(f"  imgsz: {imgsz}")
    if imgsz != 512:
        print(f"    WARNING  Expected 512, got {imgsz}")
    else:
        print(f"    OK Single-scale correctly applied")

    # Verify RandAugment
    auto_augment = args.get('auto_augment')
    print(f"  auto_augment: {auto_augment}")
    if auto_augment != "randaugment":
        print(f"    WARNING  Expected 'randaugment', got {auto_augment}")
    else:
        print(f"    OK RandAugment correctly applied")

    print()


In [ ]:
# Save completion status
completion_status = {
    "notebook": "ablation_3_counterfactual_production",
    "status": "complete",
    "timestamp": datetime.now().isoformat(),
    "seeds": RANDOM_SEEDS,
    "configuration": {
        "intervention": "Single-scale (512) + RandAugment",
        "imgsz": IMGSZ,
        "auto_augment": AUTO_AUGMENT,
        "epochs": EPOCHS,
        "patience": PATIENCE,
        "batch": BATCH,
        "lr0": LR0,
        "workers": WORKERS,
        "amp": AMP,
        "dataset": "baseline-outputs (full)"
    },
    "training_results": training_results
}

status_path = ABLATION_OUTPUT / "completion_status.json"
with open(status_path, 'w') as f:
    json.dump(completion_status, f, indent=2)

print(f"OK Completion status saved: {status_path}")
print(f"\nProduction training complete!")
print(f"Next: Download outputs and run evaluation")
